# Figure 5: Coupling of B cell differentiation and isotype fate to V(D)J gene usage

This notebook reproduces panels of **Figure 5** of the AIDA AIRR manuscript:

- **Fig. 5A** — UMAP of B cell metacells in V(D)J space colored by cell type
- **Fig. 5B** — UMAP colored by Leiden cluster
- **Fig. 5C** — Cell type composition per Leiden cluster
- **Fig. 5D** — Heat map of V/J gene segment enrichment per Leiden cluster
- **Fig. 5E** — UMAP overlay of isotype subclass distributions
- **Fig. 5F** — Pseudobulk UMAP of B cell repertoires aggregated by ethnicity x isotype
- **Fig. 5G** — Heatmap of V gene usage frequency across isotype subclasses and ethnic groups


## Imports

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats as stats
from scipy.sparse import csr_matrix

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad
import dandelion as ddl
import milopy
import milopy.core as milo
import sceleto2 as scjp

from sklearn.preprocessing import StandardScaler as standard
from sklearn.preprocessing import RobustScaler as robust
from sklearn.linear_model import Ridge

%matplotlib inline
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, color_map='OrRd')

plt.rcParams['pdf.fonttype'] = 42
sns.set_style('ticks', {"grid.color": "dimgray", "grid.linestyle": ":",
                         'axes.edgecolor': 'black', 'axes.edgewidth': 2})
sns.set_context("paper", font_scale=1.3, rc={'patch.linewidth': 1})


## Color palettes / category orders

In [ ]:
Ethnicity_order=['Chinese', 'Malay','Indian','Japanese','Korean','Thai']

anno1_order = ['B_Naive', 'B_Memory', 'B_Plasma']
anno2_order = ['B_Naive_transitional', 'B_Naive', 'B_Naive_IFN', 'B_Memory_IFN',
       'B_Memory_unswitched','B_Memory_atypical', 'B_Memory_switched', 
        'B_Plasma']


sex_colors =  ["#87CEFA", "#FB7C7C"]

age_colors = ["#E5F5E0", "#74C476", "#238B45", "#00441B"]

shm_colors = ['grey','steelblue', 'mediumblue']
 
Ethnicity_colors = ["tomato",'gold',  "dodgerblue",  "limegreen",  "aquamarine", "orchid",  'gray']

In [ ]:
anno2_colors = ['#a6cee3',
 '#1f78b4','goldenrod',
 'darkgoldenrod','lawngreen', 'saddlebrown', 
 '#33a02c',

 '#e31a1c',
]


## Load BCR-annotated AnnData and filter to QC-passing donors

In [ ]:
bdata = sc.read('data/06_250608_BDATA_IGLK_annotated_meta.h5ad')
sns.histplot(bdata.obs['PatientID'].value_counts(),bins=500)
plt.xlim(0,200)

In [ ]:
# QC: exclude donors with cell counts in lowest 5th percentile
pco = pd.DataFrame(bdata.obs['PatientID'].value_counts())
ptl = pco[pco['count'] >= np.percentile(pco['count'], 5)].index.tolist()


In [ ]:
bdata = bdata[bdata.obs['j_call_VJ_main']!='No_contig']

In [ ]:
adata = bdata[bdata.obs['PatientID'].isin(ptl)]

adata = adata[~adata.obs['Ethnicity'].isna()]

In [ ]:
adata = adata[adata.obs['Ethnicity']!='European']

In [ ]:
adata.obs['v_call_B_VDJ_main_old'] = adata.obs['v_call_B_VDJ_main'].copy()
adata.obs['v_call_B_VDJ_main'] = [a.split(',')[0] for a in adata.obs['v_call_B_VDJ_main_old']]

adata.obs['j_call_B_VDJ_main_old'] = adata.obs['j_call_B_VDJ_main'].copy()
adata.obs['j_call_B_VDJ_main'] = [a.split(',')[0] for a in adata.obs['j_call_B_VDJ_main']]

adata.obs['v_call_B_VJ_main_old'] = adata.obs['v_call_B_VJ_main'].copy()
adata.obs['v_call_B_VJ_main'] = [a.split(',')[0] for a in adata.obs['v_call_B_VJ_main_old']]

adata.obs['j_call_B_VJ_main_old'] = adata.obs['j_call_B_VJ_main'].copy()
adata.obs['j_call_B_VJ_main'] = [a.split(',')[0] for a in adata.obs['j_call_B_VJ_main_old']]

In [ ]:
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-23D']), 'IGHV3-23',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-30-3', 'IGHV3-30-5', 'IGHV3-33']), 'IGHV3-30',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV1-69D']), 'IGHV1-69',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV1-69-2']), 'IGHV1-69',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-43D']), 'IGHV3-43',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-64D']), 'IGHV3-64',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-66']), 'IGHV3-53',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV4-30-2']), 'IGHV4-30-4',  adata.obs['v_call_B_VDJ_main'])

In [ ]:
adata.obs['anno2'] = np.where(adata.obs['anno2']=='B_Plasmablast', 'B_Plasma', adata.obs['anno2'])

In [ ]:
adata.obs['anno2'] =adata.obs['anno2'].astype('category')

## Build V(D)J pseudobulk metacells (Dandelion + Milo)

We aggregate transcriptomically similar B cells into milo "metacells", then compute V/J
gene usage proportions per metacell to build the V(D)J feature space.

In [ ]:
adata = ddl.tl.setup_vdj_pseudobulk(adata,
                                    subsetby='anno2', 
                                    groups = adata.obs['anno2'].cat.categories,
                                    mode = 'B')

In [ ]:
sc.pp.neighbors(adata, use_rep = "X_pca", n_neighbors = 50)

In [ ]:
# use milo to sample neighbourhood
milo.make_nhoods(adata)
# build neighbourhood adata in adata.uns['nhood_adata']
milo.count_nhoods(adata, sample_col='PatientID') # this step is needed to build adata.uns['nhood_adata'] and sample_col can be anything
# this step is needed for plotting below
milopy.utils.build_nhood_graph(adata)
# assign neighbourhood celltype by majority voting
# results are in adata.uns['nhood_adata'].obs['nhood_annotation'] & adata.uns['nhood_adata'].obs['nhood_annotation_frac'] 
milopy.utils.annotate_nhoods(adata, anno_col='anno2') 

In [ ]:
# Prune the metacell graph to retain only edges with >=10 shared cells
min_overlap = 10
nhood_conn = adata.uns['nhood_adata'].obsp['nhood_connectivities'].todense()
nhood_conn[nhood_conn < min_overlap] = 0
adata.uns['nhood_adata'].obsp['nhood_connectivities'] = sp.sparse.csr_matrix(nhood_conn)


In [ ]:
ddd = adata.uns['nhood_adata'].copy()

In [ ]:
ddd.obs['nhood_annotation'] = ddd.obs['nhood_annotation'].astype('category')

In [ ]:
ddd.obs['nhood_annotation'] = ddd.obs['nhood_annotation'].cat.reorder_categories(anno2_order)

In [ ]:
ddd.uns['nhood_annotation_colors'] = anno2_colors

Render the milo neighborhood graph for diagnostic visualisation:

In [ ]:
# plot nhood on UMAP, but legend does not include line thickness and node size
sc.pl.embedding(ddd, basis='X_milo_graph', sizes=list(adata.uns['nhood_adata'].obs['Nhood_size']),
                color='nhood_annotation', 
                neighbors_key='nhood', frameon=False,
               ec='k', linewidth=0.3,
               title='',              
                save='Fig4_B_milo.pdf')

In [ ]:
nhood_adata = ddl.tl.vdj_pseudobulk(adata, pbs=adata.obsm["nhoods"], obs_to_take=["anno1","anno2",'c_call_VDJ','Ethnicity'],
                                  extract_cols=['v_call_B_VDJ_main', 'j_call_B_VDJ_main','v_call_B_VJ_main','j_call_B_VJ_main','c_call_VDJ_main'])

In [ ]:
nhood_adata.obs = nhood_adata.obs.join(nhood_adata.to_df()[['IGHD','IGHM','IGHG1', 'IGHG2', 'IGHG3', 'IGHG4','IGHA1','IGHA2','IGHE']])

In [ ]:
nhood_adata.var['highly_variable'] = True

Mark the constant-region (IGHC) genes as non-informative for V(D)J usage so they
are excluded from the PCA — otherwise the manifold collapses onto the dominant
isotype gradient.

In [ ]:
nhood_adata.var['highly_variable'] = np.where(nhood_adata.var.index.str.startswith('IGHG'), False, nhood_adata.var['highly_variable'])
nhood_adata.var['highly_variable'] = np.where(nhood_adata.var.index.str.startswith('IGHA'), False, nhood_adata.var['highly_variable'])
nhood_adata.var['highly_variable'] = np.where(nhood_adata.var.index == 'IGHD', False, nhood_adata.var['highly_variable'])
nhood_adata.var['highly_variable'] = np.where(nhood_adata.var.index.str.startswith('IGHM'), False, nhood_adata.var['highly_variable'])
nhood_adata.var['highly_variable'] = np.where(nhood_adata.var.index.str.startswith('IGHE'), False, nhood_adata.var['highly_variable'])

In [ ]:
nhood_adata = nhood_adata[:,nhood_adata.var['highly_variable']==True]

In [ ]:
sc.tl.pca(nhood_adata, use_highly_variable=True)
sc.pl.pca(nhood_adata, color=['anno2'])

In [ ]:
sc.pp.neighbors(nhood_adata, n_neighbors=15, n_pcs=50)
sc.tl.umap(nhood_adata, random_state=57)
sc.pl.umap(nhood_adata, color=['anno2'])

### Fig. 5A — UMAP of B cell metacells colored by cell type

In [ ]:
nhood_adata.obs['anno2'] = nhood_adata.obs['anno2'].cat.reorder_categories(anno2_order)

In [ ]:
scjp.us(nhood_adata, 'anno2', frameon=False, s=75,
         legend_fontsize=15, )
plt.title('')
scjp.save_fig('Fig5_B_total_VDJ_space_anno2','UMAP',fig_folder='figures')

### Fig. 5B — UMAP colored by Leiden cluster

In [ ]:
sc.tl.leiden(nhood_adata, resolution=0.8)

Cluster relabelling (assigns biologically meaningful order: 0–6):

In [ ]:
mapping_str = {'0':'1','1':'4','2':'5','3':'3','4':'6','5':'0','6':'2'}

# leiden labels are usually strings; cast to string category and rename
s = nhood_adata.obs['leiden'].astype(str).astype('category')
nhood_adata.obs['leiden'] = s.cat.rename_categories(mapping_str)

In [ ]:
nhood_adata.obs['leiden'] = nhood_adata.obs['leiden'].cat.reorder_categories(
    ['0', '1', '2', '3', '4', '5', '6'], ordered=True
)


In [ ]:
scjp.us(nhood_adata, 'leiden', frameon=False,s=75, palette='Accent',
       legend_loc='on data', legend_fontsize=15)
plt.title('')
scjp.save_fig('Fig5_B_total_VDJ_space_leiden','UMAP',fig_folder='figures')

### Fig. 5C — Cell type composition per Leiden cluster

In [ ]:
niso = pd.crosstab(nhood_adata.obs['leiden'],nhood_adata.obs['anno2'], normalize=0)

ax = niso.plot(kind='bar', stacked=True, 
               width=0.9,figsize=(3,5), ec='k', color=nhood_adata.uns['anno2_colors'],
              alpha=0.8)
plt.legend(loc=(1.01,0))
plt.xlabel('')
sns.despine()
plt.savefig('figures/Fig5_B_leiden_anno2_prop.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')    

### Per-leiden isotype distribution (violin facet, supplementary)

In [ ]:
df = nhood_adata.obs.copy()

# IGH variables to use as facets
isotypes = ['IGHD','IGHM','IGHG1','IGHG2','IGHG3','IGHG4','IGHA1','IGHA2','IGHE']

# (optional) fix leiden order
if hasattr(df['leiden'], 'cat'):
    x_order = list(df['leiden'].cat.categories)
else:
    x_order = sorted(df['leiden'].astype(str).unique(), key=lambda x: int(x))

# convert to long format
long = (df[isotypes]
          .assign(leiden=df['leiden'].astype(str))
          .melt(id_vars='leiden', var_name='isotype', value_name='value'))

# violin facet
g = sns.catplot(
    data=long,
    x='leiden', y='value',
    col='isotype', col_wrap=3,       # 3 per row
    kind='violin',
    order=x_order,
    cut=0, scale='width',
    inner=None,                      # remove inner box/points (replaced by strip below)
    palette='Accent',
    sharey=False,                    # set True for global comparison
    height=3.0, aspect=1.0
)

# stripplot overlay (jitter)
g.map_dataframe(
    sns.stripplot,
    x='leiden', y='value',
    order=x_order,
    jitter=0.1, size=1.5, color='k', dodge=False, alpha=0.6
)

# tidy layout
for ax in g.axes.flat:
    ax.set_xlabel('leiden')
    ax.set_ylabel('')
    for lab in ax.get_xticklabels():
        lab.set_rotation(0)
        lab.set_ha('center')

g.set_titles("{col_name}")
plt.tight_layout()

plt.savefig('figures/supple/sup6_B_leiden_iso_prop_violin.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')    
plt.show()

### Fig. 5D — Top differential V/J genes per Leiden cluster

In [ ]:
sc.tl.rank_genes_groups(test, groupby='leiden', method='wilcoxon', key_added='rgg')

# (optional) first-pass filter: loose threshold on in-group expression / fold change
# helps remove obvious housekeeping / broadly expressed genes
sc.tl.filter_rank_genes_groups(
    test, key='rgg',
    min_in_group_fraction=0.2,     # expressed in >=20% of cells in group
    max_out_group_fraction=0.05,   # <=5% in other groups
    min_fold_change=1.5            # mean expression ratio >= 1.5x
)

# 2) get results as table
df = sc.get.rank_genes_groups_df(test, group=None, key='rgg')  # all groups
# sort by score: for wilcoxon, descending 'scores' (z); for significance use ascending pvals_adj
df = df.sort_values(['group','scores'], ascending=[True, False])


In [ ]:
def unique_top_markers(df, k=5, group_col='group', gene_col='names',
                       sort_cols='scores', sort_ascending=False,
                       round_robin=True):
    # sort per group
    df2 = df.sort_values([group_col, sort_cols], ascending=[True, sort_ascending]).copy()
    groups = list(df2[group_col].unique())
    picked = {g: [] for g in groups}
    used = set()

    if round_robin:
        # round-robin: assign one per group iteratively (fair)
        cursors = {g: 0 for g in groups}
        dfs = {g: df2[df2[group_col]==g].reset_index(drop=True) for g in groups}
        done = False
        while not done:
            done = True
            for g in groups:
                while cursors[g] < len(dfs[g]) and (dfs[g].loc[cursors[g], gene_col] in used):
                    cursors[g] += 1
                if cursors[g] < len(dfs[g]) and len(picked[g]) < k:
                    gene = dfs[g].loc[cursors[g], gene_col]
                    picked[g].append(gene)
                    used.add(gene)
                    cursors[g] += 1
                    done = False
            # stop when every group has reached k
            if all(len(picked[g]) >= k or len(dfs[g]) == 0 for g in groups):
                break
    else:
        # greedy: sort across all groups, then prioritize groups still below k
        df_all = df2.sort_values(list(sort_cols), ascending=list(sort_ascending))
        for _, row in df_all.iterrows():
            g, gene = row[group_col], row[gene_col]
            if gene in used: 
                continue
            if len(picked[g]) < k:
                picked[g].append(gene)
                used.add(gene)
            if all(len(picked[gg]) >= k for gg in groups):
                break
    return picked

unique5 = unique_top_markers(df, k=5, round_robin=True)

# return as DataFrame for readability
out = (pd.DataFrame.from_dict(unique5, orient='index')
         .rename(columns=lambda i: f"top{i+1}"))
        

In [ ]:
ugenes = out.T

In [ ]:
ug = {}
for a in ugenes.columns:
    ug[a] = ugenes[a].values

In [ ]:
test.layers['scale'] = test.X.copy()
sc.pp.scale(test, layer='scale')

In [ ]:
sc.pl.matrixplot(test,ug, groupby='leiden', layer='scale', swap_axes=True, vmax=1.6, vmin=-1.6,
                 cmap='RdBu_r', ec='k',
                    figsize=(4,10), save='B_vdj_matrixplot_by_leiden_250810.pdf')

### Fig. 5E — UMAP overlay of isotype proportions on V(D)J space

In [ ]:
scjp.us(nhood_adata, 'IGHD,IGHM,IGHG1,IGHG2,IGHG3,IGHG4,IGHA1,IGHA2,IGHE'.split(','), ncols=3, frameon=False,
             s=75, vmax= 'p99')

scjp.save_fig('Fig6_B_VDJ_space_isotype_prop','UMAP',fig_folder='figures')

Optional: persist the B cell V(D)J neighborhood object.

In [ ]:
# nhood_adata.write('data/99_250930_nhood_bdata_total.h5ad')


# Fig. 5F — Pseudobulk UMAP of B cells aggregated by ethnicity x isotype

We aggregate per-cell V/J usage frequencies into pseudobulk samples defined by
(ethnicity x isotype) combinations and project onto a 2-D UMAP. The resulting layout
reveals the isotype-driven structure of B cell repertoires.

In [ ]:
import anndata as ad

In [ ]:
vdata = adata.copy()

In [ ]:
vdata.obs['Old'] = 'Young'
vdata.obs['Old'] = np.where(vdata.obs['Age'] > 40, 'Old', vdata.obs['Old'])

In [ ]:
vdata.obs['Eth_Sex_Old_iso'] = ['{}*{}*{}*{}'.format(a,b,c,d) for a,b,c,d in zip(vdata.obs['Ethnicity'],
                                                                           vdata.obs['Sex'],
                                                                           vdata.obs['Old'],
                                                                                             vdata.obs['c_call_VDJ_main'])]

In [ ]:
ighv = pd.crosstab(vdata.obs['Eth_Sex_Old_iso'],  vdata.obs['v_call_VDJ_main'], normalize=0)
ighj = pd.crosstab(vdata.obs['Eth_Sex_Old_iso'],  vdata.obs['j_call_VDJ_main'], normalize=0)
iglv = pd.crosstab(vdata.obs['Eth_Sex_Old_iso'],  vdata.obs['v_call_VJ_main'], normalize=0)
iglj = pd.crosstab(vdata.obs['Eth_Sex_Old_iso'],  vdata.obs['j_call_VJ_main'], normalize=0)

In [ ]:
vdjf = pd.concat([ighv,ighj, iglv, iglj],axis=1)

In [ ]:
vdjf = vdjf.reset_index().set_index('Eth_Sex_Old_iso')
# del vdjf['Sex']
# del vdjf['Old']

In [ ]:
vdjdata = ad.AnnData(vdjf)

In [ ]:
vdjdata.raw = vdjdata.copy()

In [ ]:
sc.pp.scale(vdjdata)

In [ ]:
sc.tl.pca(vdjdata)

Annotate the pseudobulk samples with ethnicity, isotype, sex, age group:

In [ ]:
vdjdata.obs['ethnicity'] = [a.split('*')[0] for a in vdjdata.obs.index]
vdjdata.obs['sex'] = [a.split('*')[1] for a in vdjdata.obs.index]
vdjdata.obs['old'] = [a.split('*')[2] for a in vdjdata.obs.index]
vdjdata.obs['isotype'] = [a.split('*')[3] for a in vdjdata.obs.index]


In [ ]:
sc.pp.neighbors(vdjdata,n_neighbors=10,n_pcs=10)

In [ ]:
sc.tl.umap(vdjdata)

In [ ]:
vdjdata.obs['isotype'] = vdjdata.obs['isotype'].cat.reorder_categories(['IGHD','IGHM','IGHG1','IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2', 'IGHE'])

iso_color = ['slategrey','wheat','lightskyblue','dodgerblue','navy','teal','palevioletred','lightsalmon','silver']

In [ ]:
vdjdata.uns['isotype_colors']= iso_color

In [ ]:
scjp.us(vdjdata,'isotype', frameon=False,
       ec='k', linewidth=0.5)
plt.title('')

scjp.save_fig('Fig5_B_Isotype_bulk2_umap','UMAP',fig_folder='figures')

### Alternative pseudobulk: ethnicity x isotype only (Fig. 5F)

In [ ]:
 

vdata = adata.copy()

vdata.obs['Old'] = 'Young'
vdata.obs['Old'] = np.where(vdata.obs['Age'] > 40, 'Old', vdata.obs['Old'])

vdata.obs['Eth_Sex_Old_iso'] = ['{}*{}'.format(a,b) for a,b in zip(vdata.obs['Ethnicity'],
                                                                                             vdata.obs['c_call_VDJ_main'])]

# vdata.obs['Eth_anno_iso'] = ['{}*{}*{}'.format(a,b,c) for a,b,c in zip(vdata.obs['Ethnicity'],vdata.obs['anno2'],vdata.obs['c_call_VDJ_main'])]

ighv = pd.crosstab(vdata.obs['Eth_Sex_Old_iso'],  vdata.obs['v_call_VDJ_main'], normalize=0)
ighj = pd.crosstab(vdata.obs['Eth_Sex_Old_iso'],  vdata.obs['j_call_VDJ_main'], normalize=0)
iglv = pd.crosstab(vdata.obs['Eth_Sex_Old_iso'],  vdata.obs['v_call_VJ_main'], normalize=0)
iglj = pd.crosstab(vdata.obs['Eth_Sex_Old_iso'],  vdata.obs['j_call_VJ_main'], normalize=0)

# ighv = pd.crosstab([vdata.obs['Eth_iso'],vdata.obs['Old'], vdata.obs['Sex']],  vdata.obs['v_call_VDJ_main'], normalize=0)
# ighj = pd.crosstab([vdata.obs['Eth_iso'],vdata.obs['Old'], vdata.obs['Sex']], vdata.obs['j_call_VDJ_main'], normalize=0)
# iglv = pd.crosstab([vdata.obs['Eth_iso'],vdata.obs['Old'], vdata.obs['Sex']], vdata.obs['v_call_VJ_main'], normalize=0)
# iglj = pd.crosstab([vdata.obs['Eth_iso'],vdata.obs['Old'], vdata.obs['Sex']], vdata.obs['j_call_VJ_main'], normalize=0)

vdjf = pd.concat([ighv,ighj, iglv, iglj],axis=1)

vdjf = vdjf.reset_index().set_index('Eth_Sex_Old_iso')
# del vdjf['Sex']
# del vdjf['Old']

vdjdata = ad.AnnData(vdjf)

# veta = vdata.obs.drop_duplicates('Eth_iso')[['Eth_iso','Old','Sex']].set_index('Eth_iso')
# vdjdata.obs = vdjdata.obs.merge(veta, left_index=True, right_index=True)

vdjdata.raw = vdjdata.copy()

sc.pp.scale(vdjdata)

sc.tl.pca(vdjdata)

# vdjdata.obs['anno2'] = [a.split('*')[1] for a in vdjdata.obs.index]
vdjdata.obs['Ethnicity'] = [a.split('*')[0] for a in vdjdata.obs.index] 
# vdjdata.obs['Sex'] = [a.split('*')[1] for a in vdjdata.obs.index]
# vdjdata.obs['Old'] = [a.split('*')[2] for a in vdjdata.obs.index]
# vdjdata.obs['anno2'] = [a.split('*')[-2] for a in vdjdata.obs.index]
vdjdata.obs['isotype'] = [a.split('*')[-1] for a in vdjdata.obs.index]

sc.pl.pca(vdjdata, color='isotype', components=(1,2))

sc.pp.neighbors(vdjdata,n_neighbors=10,n_pcs=10)

sc.tl.umap(vdjdata)

vdjdata.obs['isotype'] = vdjdata.obs['isotype'].cat.reorder_categories(['IGHD','IGHM','IGHG1','IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2', 'IGHE'])

iso_color = ['slategrey','wheat','lightskyblue','dodgerblue','navy','teal','palevioletred','lightsalmon','silver']

vdjdata.uns['isotype_colors']= iso_color

scjp.us(vdjdata,'isotype', frameon=False,
       ec='k', linewidth=0.5)
plt.title('')

scjp.save_fig('Fig5_B_Isotype_bulk1_umap','UMAP',fig_folder='figures')

# Fig. 5G — V gene usage heatmap across isotype subclasses and ethnic groups

For each donor and isotype, compute IGHV usage proportions and average per
(isotype, ethnicity). Standard-scale across IGHV genes to highlight isotype-specific
preferences while preserving ethnicity-level variation.

In [ ]:
vdata = adata.copy()
# [adata.obs['anno1']!='B_Naive']

In [ ]:
vdata.obs['c_call_VDJ'] = vdata.obs['c_call_VDJ'].cat.reorder_categories(['IGHD','IGHM','IGHG1','IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2', 'IGHE'])

In [ ]:
meta = vdata.obs.drop_duplicates('PatientID').set_index('PatientID')[['Age','Sex','BMI','Country','Ethnicity']]

In [ ]:
tf = vdata.obs[(vdata.obs['c_call_VDJ']!='IGHE')] 
tf['c_call_VDJ'] = tf['c_call_VDJ'].cat.remove_unused_categories()
ttf = pd.crosstab([tf['c_call_VDJ'],tf['PatientID']], tf['v_call_B_VDJ_main'], normalize=0)

ttf = ttf[ttf.mean()[ttf.mean()>0.005].index]

mf = ttf.copy()

mf = mf.reset_index().merge(meta.reset_index()[['PatientID','Ethnicity']], left_on='PatientID', right_on='PatientID',how='left' )

mmf = mf.groupby(['c_call_VDJ','Ethnicity']).mean()

mmf = pd.DataFrame(standard().fit_transform(mmf.T), index = mmf.T.index, columns=mmf.T.columns)
mmf = mmf.T 

plt.figure(figsize=(13,13))
fig = sns.heatmap(mmf,cmap='vlag', vmax=2.6, vmin=-2.6, linecolor='k', linewidth=0.3,
                  cbar_kws={"shrink": 0.5})
cbar = fig.figure.get_children()[-1]
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")
cbar.set_ylabel('Standard scale', size=15)
 
plt.xlabel('')
plt.ylabel('')


fig.axhline(y = 0, color='k',linewidth = 3) 
fig.axhline(y = mmf.shape[0], color = 'k', 
            linewidth = 3) 

fig.axvline(x = 0, color = 'k', 
            linewidth = 3) 

fig.axvline(x = mmf.shape[1],  
             color = 'k', linewidth = 3) 

fig.axhline(y = 6, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 12, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 18, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 24, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 30, color = 'k', 
            linewidth = 1.5, linestyle='--') 

fig.axhline(y = 36, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 42, color = 'k', 
            linewidth = 1.5, linestyle='--') 

# fig.axhline(y = 48, color = 'k', 
#             linewidth = 2, linestyle='--') 


plt.savefig('figures/Fig5_B_isotype_IGHV_heatmap.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')    

### Top variable IGHV genes (per isotype, ranked by variance) — point plot

In [ ]:
ttf = pd.crosstab(
    tf['c_call_VDJ'],
    tf['v_call_B_VDJ_main'],
    normalize=0   # per-row (equivalent to normalize='index')
)

# (2) variance per column (v_call_B_VDJ_main)
variance_per_vdj = ttf.var(axis=0)

# (3) sort by descending variance
variance_sorted = variance_per_vdj.sort_values(ascending=False)

In [ ]:
tf = vdata.obs[(vdata.obs['c_call_VDJ']!='IGHE')] 
tf['c_call_VDJ'] = tf['c_call_VDJ'].cat.remove_unused_categories()
ttf = pd.crosstab([tf['c_call_VDJ'],tf['PatientID']], tf['v_call_B_VDJ_main'], normalize=0)

ttf = ttf[ttf.mean()[ttf.mean()>0.005].index]

mf = ttf.copy()

mdf = mf.reset_index().merge(meta.reset_index()[['PatientID','Ethnicity']], left_on='PatientID', right_on='PatientID',how='left' )
 

In [ ]:
df_long = mdf.melt(
    id_vars=['c_call_VDJ', 'Ethnicity'],
    value_vars=variance_sorted[:9].index,
    var_name='gene',
    value_name='proportion'
)

# 2) build facet grid via catplot
g = sns.catplot(
    data=df_long,
    x='c_call_VDJ', y='proportion',
    hue='Ethnicity',
    col='gene',
    col_wrap=3,               # 3 columns; total subplots arranged in rows of 3
    kind='point',
    hue_order=Ethnicity_order,
    palette=Ethnicity_colors,
    dodge=0.5,                # spacing between errorbars
    capsize=0.1,              # errorbar cap length
    errwidth=1,               # errorbar line width
    sharey=False,              # independent y-scale per facet
     errorbar='se',
    height=3.5,        # subplot height (inches)
    aspect=0.6,  
)

# 3) format each axis
for ax in g.axes.flatten():
    sns.despine(ax=ax)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=90)   # rotate x-axis labels

# 4) tighten layout and place legend
g.fig.tight_layout()
# g.add_legend(title='Ethnicity', bbox_to_anchor=(1, 0.5), loc='center left')
plt.legend(loc=(1.01,0))
plt.show()